<a href="https://colab.research.google.com/github/Hossain-Mahmud00/cantera-projects/blob/main/Heat_Exchanger_Sizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install thermo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.8/608.8 kB 29.8 MB/s eta 0:00:00


In [3]:

from thermo import Chemical
import numpy as np
import matplotlib.pyplot as plt

tcs = Chemical('10025-78-2', T=305, P=101325)
m_tcs = 1.0                 # kg/s of TCS vapor to condense
Q = m_tcs * tcs.Hvap        # heat duty in W
print('Hvap (kJ/kg):', tcs.Hvap / 1000)
print('Duty Q (kW):', Q / 1000)

w = Chemical('water', T=308)
Tin, Tout = 303.15, 313.15  # cooling water 30 to 40 C
m_w = Q / (w.Cp * (Tout - Tin))
print('Cooling water (kg/s):', m_w)

Hvap (kJ/kg): 183.14561239153144
Duty Q (kW): 183.14561239153144
Cooling water (kg/s): 4.38225026639426


In [4]:

Tc = 305.0                      # TCS condensing temperature (K), about 32 C
Tin, Tout = 278.15, 288.15      # chilled water 5 to 15 C
w = Chemical('water', T=283)
m_w = Q / (w.Cp * (Tout - Tin))
print('Chilled water (kg/s):', m_w)

dT1, dT2 = Tc - Tin, Tc - Tout
LMTD = (dT1 - dT2) / np.log(dT1 / dT2)
print('LMTD (K):', LMTD)

for U in [300, 500, 800]:
    print('U =', U, ' Area (m2):', round(Q / (U * LMTD), 1))

Chilled water (kg/s): 4.365391391593883
LMTD (K): 21.463134856544325
U = 300  Area (m2): 28.4
U = 500  Area (m2): 17.1
U = 800  Area (m2): 10.7


In [5]:

import math
tl = Chemical('10025-78-2', T=300, P=101325)
print('TCS liquid:', tl.phase, tl.rho, tl.kl, tl.mu)

g = 9.81
do, di = 0.01905, 0.01575      # tube OD, ID (m): 3/4 inch tube
v = 1.5                        # water speed in tubes (m/s)

# tube side (chilled water): Dittus-Boelter
Re = w.rho*v*di/w.mu
Pr = w.Cp*w.mu/w.kl
hi = 0.023*Re**0.8*Pr**0.4*w.kl/di
print('Re:', round(Re), ' Pr:', round(Pr,1), ' hi:', round(hi))

# outside (TCS condensing on a horizontal tube): Nusselt
dTf = 8.0                      # assumed film temperature drop (K)
ho = 0.725*(tl.kl**3*tl.rho**2*g*tcs.Hvap/(tl.mu*do*dTf))**0.25
print('ho:', round(ho))

# overall U based on outside area
kw, Rfi, Rfo = 16.0, 2e-4, 1e-4
Uo = 1/(1/ho + Rfo + do*math.log(do/di)/(2*kw) + Rfi*do/di + do/(di*hi))
A = Q/(Uo*LMTD)
m_tube = w.rho*v*math.pi*di**2/4
N = math.ceil(m_w/m_tube)
L = A/(N*math.pi*do)
print('U (W/m2K):', round(Uo), ' Area (m2):', round(A,1))
print('Tubes:', N, ' Length per tube (m):', round(L,1))

TCS liquid: l 1331.0432651581905 0.08517770965617234 0.00030471214481933735
Re: 18012  Pr: 9.7  hi: 5214
ho: 1850
U (W/m2K): 814  Area (m2): 10.5
Tubes: 15  Length per tube (m): 11.7


In [6]:

dTf = 8.0
for i in range(20):
    ho = 0.725*(tl.kl**3*tl.rho**2*g*tcs.Hvap/(tl.mu*do*dTf))**0.25
    Uo = 1/(1/ho + Rfo + do*math.log(do/di)/(2*kw) + Rfi*do/di + do/(di*hi))
    dTf = Uo*LMTD/ho

A = Q/(Uo*LMTD)
passes = 2
L = A/(N*passes*math.pi*do)
f = 0.316*Re**-0.25
dP = f*(passes*L/di)*0.5*w.rho*v**2

print('Film drop (K):', round(dTf,1), ' ho:', round(ho), ' U:', round(Uo))
print('Area (m2):', round(A,1))
print('Tubes per pass:', N, ' passes:', passes, ' total tubes:', N*passes)
print('Tube length (m):', round(L,2))
print('Water side pressure drop (bar):', round(dP/1e5,2))

Film drop (K): 9.7  ho: 1762  U: 797
Area (m2): 10.7
Tubes per pass: 15  passes: 2  total tubes: 30
Tube length (m): 5.96
Water side pressure drop (bar): 0.23
